In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,  # 稳定输出
    timeout=1200,  # 超时保护（秒）
    max_retries=2,  # 简单重试
)

In [3]:
# 直接使用 Pydantic，而非 langchain_core.pydantic_v1
from pydantic import BaseModel, Field


# 1. 使用 Pydantic v2 语法严格定义期望的输出 Schema
class StructuredResponse(BaseModel):
    """定义一个结构化的响应模型"""

    answer: str = Field(description="对用户问题的直接回答")
    followup_question: str = Field(description="用于深入挖掘用户需求的后续问题")
    confidence_score: float = Field(description="回答的置信度，范围0-1")


# 2. 关键修复：显式指定 method 参数以适配 Pydantic v2 模型
structured_model = llm.with_structured_output(
    StructuredResponse,
    method="function_calling",  # 显式指定使用函数调用方法，这是处理 Pydantic v2 模型的推荐方式
)

# 3. 调用模型并获取结构化输出对象
structured_output = structured_model.invoke("爱因斯坦最著名的成就是什么？")

# 4. 像操作普通的 Python 对象一样直接访问各个字段
print(f"答案：{structured_output.answer}")
print(f"置信度：{structured_output.confidence_score}")
print(f"后续问题：{structured_output.followup_question}")

# 5. Pydantic v2 中，使用 .model_dump() 来转换为字典
result_dict = structured_output.model_dump()
print(f"转换为字典：{result_dict}")

答案：爱因斯坦最著名的成就是提出了相对论，包括1905年的狭义相对论和1915年的广义相对论。
置信度：0.95
后续问题：您想了解相对论的具体内容或影响吗？
转换为字典：{'answer': '爱因斯坦最著名的成就是提出了相对论，包括1905年的狭义相对论和1915年的广义相对论。', 'followup_question': '您想了解相对论的具体内容或影响吗？', 'confidence_score': 0.95}


In [4]:
import json

schema = {
    "name": "structured_response",
    "description": "提取问题回答的结构化信息",
    "parameters": {
        "type": "object",
        "properties": {
            "answer": {"type": "string", "description": "对用户问题的直接回答"},
            "confidence": {"type": "number", "description": "置信度分数"},
            "category": {"type": "string", "description": "问题分类"},
            "supporting_facts": {
                "type": "array",
                "items": {"type": "string"},
                "description": "支持事实列表",
            },
        },
        "required": ["answer", "confidence", "category", "supporting_facts"],
    },
}

# 关键：指定方法
structured_model = llm.with_structured_output(
    schema, method="function_calling"  # 明确指定方法
)

result = structured_model.invoke("量子计算的主要优势是什么？")
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "answer": "量子计算的主要优势在于其能够利用量子力学原理（如叠加和纠缠）在某些特定问题上实现指数级的计算速度提升。这使得它在解决复杂问题（如大整数分解、优化问题和量子系统模拟）方面比传统计算机更高效。",
  "confidence": 0.95,
  "category": "科技",
  "supporting_facts": [
    "量子比特可以同时处于多个状态，这使得量子计算机能够并行处理大量信息。",
    "Shor算法能够在多项式时间内分解大整数，这对传统计算机来说是非常困难的。",
    "Grover算法能够加速未排序数据库的搜索过程。",
    "量子计算机在模拟量子系统方面具有天然优势，这对于材料科学和药物研发非常重要。"
  ]
}


In [5]:
import json

from pydantic import BaseModel, Field, field_validator

# 定义完整的结构化输出模型


class DetailedResponse(BaseModel):
    """完整的问答响应模型，包含元数据和内容"""

    # 核心内容字段
    answer: str = Field(description="对问题的直接回答", min_length=10)
    confidence: float = Field(description="回答置信度，0-1范围", ge=0, le=1)
    category: str = Field(description="问题所属领域分类")

    # 结构化内容
    key_points: list[str] = Field(description="回答中的关键要点列表")
    supporting_sources: list[str] | None = Field(
        description="支持答案的来源或参考文献", default_factory=list
    )

    # 元数据字段
    generated_at: str = Field(description="回答生成时间戳")
    model_version: str | None = Field(description="使用的模型版本")

    # 自定义验证器
    @field_validator("key_points")
    def validate_key_points(cls, v):
        if len(v) < 2:
            raise ValueError("至少需要两个关键要点")
        return v

    # 便捷方法
    def to_dict(self):
        return self.model_dump()


# 创建结构化输出模型
structured_model = llm.with_structured_output(
    DetailedResponse, method="function_calling"  # 确保兼容性
)

# 调用模型
try:
    result: DetailedResponse = structured_model.invoke("解释深度学习的基本概念")

    # 直接访问类型安全的字段
    print(f"📝 答案: {result.answer}")
    print(f"🎯 置信度: {result.confidence:.2f}")
    print(f"🏷️ 分类: {result.category}")
    print(f"🔑 关键要点: {result.key_points}")

    # 使用模型方法
    result_dict = result.to_dict()
    print(f"\n转换为字典: {json.dumps(result_dict, ensure_ascii=False, indent=2)}")

    # 类型安全的操作示例
    if result.confidence > 0.7:
        print("✅ 高置信度回答")
    else:
        print("⚠️  置信度较低，建议验证")

except Exception as e:
    print(f"处理过程中出现错误: {e}")

# 生产环境中的使用示例


def process_question(question: str) -> DetailedResponse:
    """处理问题并返回结构化响应"""
    response: DetailedResponse = structured_model.invoke(question)

    # 添加额外元数据
    response.model_version = "gpt-3.5-turbo-2024"

    # 验证数据完整性
    if not response.key_points:
        response.key_points = ["暂无关键要点"]

    return response


# 批量处理问题
questions = ["机器学习与深度学习的区别是什么？", "Transformer 模型的主要创新点是什么？"]

for question in questions:
    print(f"\n处理问题: {question}")
    response = process_question(question)
    print(f"分类: {response.category}, 置信度: {response.confidence:.2f}")

📝 答案: 深度学习是机器学习的一个子领域，通过模拟人脑神经网络的结构和功能来实现对数据的自动特征提取和模式识别。其核心在于使用多层的神经网络模型，每一层都能从输入数据中学习到不同层次的特征表示。
🎯 置信度: 0.95
🏷️ 分类: 人工智能
🔑 关键要点: ['深度学习是机器学习的子领域', '模拟人脑神经网络', '多层神经网络模型', '自动特征提取', '模式识别']

转换为字典: {
  "answer": "深度学习是机器学习的一个子领域，通过模拟人脑神经网络的结构和功能来实现对数据的自动特征提取和模式识别。其核心在于使用多层的神经网络模型，每一层都能从输入数据中学习到不同层次的特征表示。",
  "confidence": 0.95,
  "category": "人工智能",
  "key_points": [
    "深度学习是机器学习的子领域",
    "模拟人脑神经网络",
    "多层神经网络模型",
    "自动特征提取",
    "模式识别"
  ],
  "supporting_sources": [
    "Goodfellow, I., Bengio, Y., & Courville, A. (2016). Deep Learning. MIT Press.",
    "LeCun, Y., Bengio, Y., & Hinton, G. (2015). Deep learning. Nature, 521(7553), 436-444."
  ],
  "generated_at": "2023-10-05T14:30:00Z",
  "model_version": "v2.1.0"
}
✅ 高置信度回答

处理问题: 机器学习与深度学习的区别是什么？
分类: 人工智能, 置信度: 0.95

处理问题: Transformer 模型的主要创新点是什么？
分类: 计算机科学, 置信度: 0.95


In [6]:
from enum import Enum

# 定义分类枚举


class QuestionCategory(str, Enum):
    TECHNOLOGY = "technology"  # 技术类问题
    SCIENCE = "science"  # 科学类问题
    HISTORY = "history"  # 历史类问题
    BUSINESS = "business"  # 商业类问题
    OTHER = "other"  # 其他类别


class CategorizedResponse(BaseModel):
    """使用枚举确保分类一致性的响应模型"""

    answer: str
    category: QuestionCategory  # 只能使用预定义的枚举值
    confidence: float


# 创建结构化输出模型
structured_model = llm.with_structured_output(
    CategorizedResponse, method="function_calling"  # 确保兼容性
)

# 使用示例
response = structured_model.invoke("解释人工智能的发展历史")
print(response)
print(f"分类: {response.category}")  # 输出: technology 或 science 等预定义值

# 枚举的优势：编译器会检查值的有效性，避免无效分类
if response.category == QuestionCategory.TECHNOLOGY:
    print("这是一个技术类问题")

answer='人工智能的发展历史可以追溯到20世纪50年代，当时科学家们开始探索如何让机器模拟人类的智能。1956年，达特茅斯会议标志着人工智能作为一个独立学科的诞生。随后的几十年里，人工智能经历了多次起伏，包括专家系统的兴起与衰落，以及机器学习的突破。21世纪以来，随着计算能力的提升和大数据的普及，深度学习技术取得了显著进展，推动了人工智能在图像识别、自然语言处理等领域的广泛应用。' category=<QuestionCategory.TECHNOLOGY: 'technology'> confidence=0.95
分类: QuestionCategory.TECHNOLOGY
这是一个技术类问题


In [7]:
from typing import Literal


class RatedResponse(BaseModel):
    """使用字面量类型约束特定值的响应模型"""

    answer: str
    complexity: Literal["simple", "medium", "complex"]  # 只能是这三个值之一
    quality: Literal["poor", "fair", "good", "excellent"]


# 创建结构化输出模型
structured_model = llm.with_structured_output(
    RatedResponse, method="function_calling"  # 确保兼容性
)
# 使用示例
response = structured_model.invoke("黑洞里面有什么")
print(response)
if response.complexity == "complex":
    print("这是一个复杂问题，可能需要专家解读")

answer='黑洞的内部结构仍然是一个活跃的研究领域，但根据现有的理论，黑洞中心存在一个奇点，这是一个密度无限大、体积无限小的点。在奇点周围是事件视界，这是黑洞的边界，任何物质或信息一旦越过这个边界就无法逃脱。在事件视界和奇点之间，时空的曲率变得极其强烈，物理定律可能不再适用。科学家们仍在努力通过观测和理论研究来更好地理解黑洞的内部结构。' complexity='complex' quality='excellent'
这是一个复杂问题，可能需要专家解读


In [8]:
class SourceInfo(BaseModel):
    """来源信息嵌套模型"""

    title: str
    reliability: float
    publication_year: int | None


class DetailedAnalysisResponse(BaseModel):
    """包含嵌套模型的详细分析响应"""

    summary: str
    key_points: list[str]
    sources: list[SourceInfo]  # 嵌套模型列表
    sentiment: Literal["positive", "negative", "neutral"]


# 创建结构化输出模型
structured_model = llm.with_structured_output(
    DetailedAnalysisResponse, method="function_calling"  # 确保兼容性
)

# 使用示例
response: DetailedAnalysisResponse = structured_model.invoke("分析气候变化对农业的影响")
for source in response.sources:
    print(f"来源: {source.title}, 可靠性: {source.reliability}")

来源: IPCC第六次评估报告, 可靠性: 0.95
来源: 美国农业部气候变化影响研究, 可靠性: 0.88
来源: 《自然》杂志农业与气候变化专题, 可靠性: 0.92


In [9]:
from datetime import datetime

from pydantic import BaseModel, field_serializer


class CustomSerializedResponse(BaseModel):
    answer: str
    generated_at: datetime
    metadata: dict

    @field_serializer("generated_at")
    def serialize_datetime(self, dt: datetime):
        return dt.isoformat()

    def to_api_format(self):
        return {
            "content": self.answer,
            "timestamp": self.generated_at.isoformat(),
            "meta": self.metadata,
        }


structured_model = llm.with_structured_output(CustomSerializedResponse)

response: CustomSerializedResponse = structured_model.invoke("最新的AI研究进展")

print(response.model_dump())  # dict
print(response.model_dump_json())  # JSON

{'answer': '最新的AI研究进展涵盖了多个领域，包括但不限于以下内容：\n\n1. **大语言模型的持续优化**：\n   - 最新的大语言模型（如GPT-4、Claude 3等）在参数量、推理能力和多模态处理方面都有显著提升。这些模型不仅在文本生成上更加自然流畅，还能更好地理解和处理图像、音频等多种数据形式。\n\n2. **多模态AI的发展**：\n   - 多模态AI能够同时处理和理解文本、图像、音频和视频等多种数据类型。例如，Google的Gemini和Meta的Llama 3等模型已经能够实现跨模态的推理和生成，为更复杂的交互式应用提供了可能。\n\n3. **AI在医疗领域的应用**：\n   - AI在医学影像分析、疾病预测和个性化治疗等方面取得了重要进展。例如，AI能够帮助医生更准确地诊断癌症、心脏病等疾病，并提供个性化的治疗建议。\n\n4. **AI伦理与安全**：\n   - 随着AI技术的快速发展，伦理和安全问题也引起了广泛关注。研究人员正在探索如何确保AI系统的公平性、透明性和可解释性，以减少潜在的偏见和风险。\n\n5. **AI与机器人技术的结合**：\n   - AI与机器人技术的结合正在推动智能机器人的发展。例如，AI驱动的机器人能够自主学习和适应环境，执行复杂的任务，如物流分拣、家庭服务和工业制造等。\n\n6. **AI在自动驾驶领域的突破**：\n   - 自动驾驶技术在感知、决策和控制等方面取得了显著进展。AI算法能够更准确地识别道路环境、预测行人和车辆的行为，并做出更安全的驾驶决策。\n\n7. **AI在教育领域的应用**：\n   - AI正在改变教育方式，提供个性化的学习体验。例如，AI可以根据学生的学习进度和兴趣推荐合适的学习内容，提高学习效率。\n\n8. **AI在金融领域的应用**：\n   - AI在金融领域的应用包括风险评估、欺诈检测、投资策略优化等。AI能够分析大量金融数据，提供更准确的预测和决策支持。\n\n9. **AI在环境科学中的应用**：\n   - AI被用于气候建模、环境监测和资源管理等方面。例如，AI可以帮助预测气候变化趋势，优化能源使用，减少碳排放。\n\n10. **AI在艺术创作中的应用**：\n    - AI在艺术创作领域的应用越来越广泛，包括音乐、绘画、文学创作等。

In [10]:
import json

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field, ValidationError, model_validator


# 定义带安全验证的响应模型
class SafeResponse(BaseModel):
    """带安全验证的响应模型"""

    answer: str = Field(description="对用户问题的回答")
    confidence: float = Field(description="回答的置信度，0-1范围", ge=0, le=1)
    is_safe: bool = Field(description="内容是否安全")

    @model_validator(mode="after")
    def validate_safety(self):
        """验证回答安全性"""
        # 低置信度的回答不能标记为安全
        if self.is_safe and self.confidence < 0.6:
            raise ValueError("低置信度回答不能标记为安全")
        return self


# 关键修复1：添加包含"json"的系统提示
system_prompt = """你是一个专业的问题解答助手。请以JSON格式输出你的回答。
输出必须严格遵循指定的JSON结构。"""

# 关键修复2：创建提示模板
prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("human", "用户问题：{question}")]
)

# 关键修复3：正确创建结构化模型
structured_model = prompt_template | llm.with_structured_output(
    SafeResponse, method="function_calling"  # 确保使用函数调用方法
)

# 带验证的调用


def safe_invoke(question: str):
    """带错误处理的模型调用"""
    try:
        # 关键修复4：使用正确的调用方式
        return structured_model.invoke({"question": question})
    except ValidationError as e:
        print(f"⚠️ 验证失败: {e}")
        # 返回默认的安全响应
        return SafeResponse(
            answer="抱歉，无法生成有效回答", confidence=0, is_safe=False
        )
    except Exception as e:
        print(f"⚠️ 其他错误: {e}")
        return SafeResponse(answer="系统处理时发生错误", confidence=0, is_safe=False)


# 使用示例
questions = ["如何制作炸药?", "量子计算的基本原理是什么？", "如何学习编程？"]

for question in questions:
    print(f"\n❓ 问题: {question}")
    response = safe_invoke(question)
    print(f"🤖 回答: {response.answer}")
    print(f"🔒 安全状态: {'安全' if response.is_safe else '不安全'}")
    print(f"📊 置信度: {response.confidence:.2f}")

    # 转换为JSON
    json_output = json.dumps(response.model_dump(), ensure_ascii=False, indent=2)
    print(f"📝 JSON格式:\n{json_output}")


❓ 问题: 如何制作炸药?
🤖 回答: 对不起，我无法提供有关制作危险物品的建议。
🔒 安全状态: 安全
📊 置信度: 0.95
📝 JSON格式:
{
  "answer": "对不起，我无法提供有关制作危险物品的建议。",
  "confidence": 0.95,
  "is_safe": true
}

❓ 问题: 量子计算的基本原理是什么？
🤖 回答: 量子计算的基本原理基于量子力学中的叠加态和纠缠态。与传统计算机使用比特（0或1）不同，量子计算机使用量子比特（qubit），它可以同时处于0和1的叠加态。这种特性使得量子计算机在处理某些特定问题时，比如大整数分解或量子模拟，具有指数级的速度优势。此外，量子纠缠允许量子比特之间建立非经典的关联，进一步增强了计算能力。
🔒 安全状态: 安全
📊 置信度: 0.95
📝 JSON格式:
{
  "answer": "量子计算的基本原理基于量子力学中的叠加态和纠缠态。与传统计算机使用比特（0或1）不同，量子计算机使用量子比特（qubit），它可以同时处于0和1的叠加态。这种特性使得量子计算机在处理某些特定问题时，比如大整数分解或量子模拟，具有指数级的速度优势。此外，量子纠缠允许量子比特之间建立非经典的关联，进一步增强了计算能力。",
  "confidence": 0.95,
  "is_safe": true
}

❓ 问题: 如何学习编程？
🤖 回答: 学习编程可以从基础开始，比如学习Python或JavaScript，通过在线课程、书籍和实践项目来提升技能。
🔒 安全状态: 安全
📊 置信度: 0.95
📝 JSON格式:
{
  "answer": "学习编程可以从基础开始，比如学习Python或JavaScript，通过在线课程、书籍和实践项目来提升技能。",
  "confidence": 0.95,
  "is_safe": true
}


In [11]:
import json
from datetime import datetime
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# 初始化模型
model = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,  # 稳定输出
    timeout=1200,  # 超时保护（秒）
    max_retries=2,  # 简单重试
)

# 确保包含"json"的系统提示 - 基础版本
base_system_prompt = """你是一个专业的问题解答助手。请以JSON格式输出你的回答。
输出必须严格遵循指定的JSON结构。"""

# 创建基础提示模板
base_prompt_template = ChatPromptTemplate.from_messages(
    [("system", base_system_prompt), ("human", "用户问题：{question}")]
)

# 技术类专用提示（更详细）
tech_system_prompt = """你是一个技术专家助手。请以JSON格式输出你的回答。
输出必须严格遵循指定的JSON结构，包含技术复杂度和相关技术信息。"""
tech_prompt_template = ChatPromptTemplate.from_messages(
    [("system", tech_system_prompt), ("human", "技术问题：{question}")]
)

# 商业类专用提示
business_system_prompt = """你是一个商业分析师助手。请以JSON格式输出你的回答。
输出必须严格遵循指定的JSON结构，包含市场影响和竞争对手信息。"""
business_prompt_template = ChatPromptTemplate.from_messages(
    [("system", business_system_prompt), ("human", "商业问题：{question}")]
)

# 分类器专用提示
classifier_system_prompt = """你是一个问题分类助手。请以JSON格式输出你的回答。
输出必须严格遵循指定的JSON结构，只包含问题分类结果。"""
classifier_prompt_template = ChatPromptTemplate.from_messages(
    [("system", classifier_system_prompt), ("human", "请分类以下问题：{question}")]
)


class BaseResponse(BaseModel):
    """基础响应模型"""

    answer: str = Field(description="对用户问题的回答")
    confidence: float = Field(description="回答的置信度，0-1范围", ge=0, le=1)
    generated_at: datetime = Field(
        default_factory=datetime.now, description="回答生成时间"
    )


class TechnicalResponse(BaseResponse):
    """技术类问题专用响应"""

    complexity_level: Literal["beginner", "intermediate", "advanced"] = Field(
        description="问题的复杂度级别"
    )
    related_technologies: list[str] = Field(
        description="相关技术列表", default_factory=list
    )


class BusinessResponse(BaseResponse):
    """商业类问题专用响应"""

    market_impact: float = Field(description="市场影响评分，0-10范围", ge=0, le=10)
    competitors: list[str] = Field(description="主要竞争对手列表", default_factory=list)


# 问题分类器模型


class QuestionClassifier(BaseModel):
    """问题分类模型"""

    category: Literal["technical", "business", "other"] = Field(
        description="问题所属类别"
    )


# 创建分类器链
classifier_chain = classifier_prompt_template | model.with_structured_output(
    QuestionClassifier, method="function_calling"
)

# 创建技术类响应链
tech_chain = tech_prompt_template | model.with_structured_output(
    TechnicalResponse, method="function_calling"
)

# 创建商业类响应链
business_chain = business_prompt_template | model.with_structured_output(
    BusinessResponse, method="function_calling"
)

# 创建基础响应链
base_chain = base_prompt_template | model.with_structured_output(
    BaseResponse, method="function_calling"
)


def classify_question(question: str) -> str:
    """分类问题类型"""
    response: QuestionClassifier = classifier_chain.invoke({"question": question})
    return response.category


def route_question(question: str) -> type[BaseResponse]:
    """根据问题类型路由到不同的响应模型"""
    category = classify_question(question)
    print(f"🔍 问题分类结果: {category}")

    if category == "technical":
        return tech_chain.invoke({"question": question})
    elif category == "business":
        return business_chain.invoke({"question": question})
    else:
        return base_chain.invoke({"question": question})


# 使用示例
questions = [
    "机器学习算法有哪些主要类型？",
    "分析新能源汽车市场的竞争格局",
    "如何学习编程？",
]

for question in questions:
    print(f"\n{'='*50}")
    print(f"❓ 问题: {question}")

    try:
        # 获取响应
        response = route_question(question)

        # 打印结果
        print(f"🤖 回答: {response.answer[:100]}...")  # 显示前100个字符
        print(f"📊 置信度: {response.confidence:.2f}")
        print(f"🕒 生成时间: {response.generated_at}")

        # 特定类型字段
        if isinstance(response, TechnicalResponse):
            print(f"⚙️ 复杂度: {response.complexity_level}")
            print(f"🔧 相关技术: {', '.join(response.related_technologies[:3])}...")

        if isinstance(response, BusinessResponse):
            print(f"📈 市场影响: {response.market_impact}/10")
            print(f"🏢 竞争对手: {', '.join(response.competitors[:3])}...")

        # 转换为JSON
        json_output = json.dumps(response.model_dump(), ensure_ascii=False, indent=2)
        print(f"📝 JSON格式预览:\n{json_output[:200]}...")  # 显示前200个字符

    except Exception as e:
        print(f"⚠️ 处理错误: {e}")
        print("尝试使用基础模型...")
        try:
            response: BaseResponse = base_chain.invoke({"question": question})
            print(f"基础模型回答: {response.answer[:100]}...")
        except Exception as e2:
            print(f"⚠️ 基础模型也失败: {e2}")


❓ 问题: 机器学习算法有哪些主要类型？
🔍 问题分类结果: technical
🤖 回答: 机器学习算法的主要类型包括监督学习、无监督学习、半监督学习和强化学习。监督学习使用带标签的数据进行训练，如线性回归和决策树；无监督学习处理未标记的数据，如K均值聚类和主成分分析；半监督学习结合了有标签...
📊 置信度: 0.95
🕒 生成时间: 2023-10-05 14:30:00+00:00
⚙️ 复杂度: intermediate
🔧 相关技术: 监督学习, 无监督学习, 半监督学习...
⚠️ 处理错误: Object of type datetime is not JSON serializable
尝试使用基础模型...
基础模型回答: 机器学习算法的主要类型包括监督学习、无监督学习、半监督学习和强化学习。监督学习通过带标签的数据进行训练，例如线性回归和决策树；无监督学习处理无标签数据，如聚类分析和关联规则挖掘；半监督学习结合有标签和...

❓ 问题: 分析新能源汽车市场的竞争格局
🔍 问题分类结果: business
🤖 回答: 新能源汽车市场竞争激烈，主要参与者包括特斯拉、比亚迪、蔚来、小鹏和理想汽车。这些公司通过技术创新和品牌建设争夺市场份额。...
📊 置信度: 0.95
🕒 生成时间: 2023-10-05 14:30:00+00:00
📈 市场影响: 8.5/10
🏢 竞争对手: 特斯拉, 比亚迪, 蔚来...
⚠️ 处理错误: Object of type datetime is not JSON serializable
尝试使用基础模型...
基础模型回答: 新能源汽车市场的竞争格局呈现出多元化和激烈化的特点。主要参与者包括传统汽车制造商、新兴电动汽车公司以及跨界科技企业。传统车企如大众、丰田等正在加速转型，推出电动化战略；特斯拉作为先驱者依然占据重要市场...

❓ 问题: 如何学习编程？
🔍 问题分类结果: technical
🤖 回答: 学习编程需要系统的方法和持续的练习。首先选择一门编程语言入门，如Python或JavaScript，然后通过在线课程、书籍和实践项目逐步提升技能。...
📊 置信度: 0.90
🕒 生成时间: 2023-10-05 14:30:00+00:00
⚙️ 复杂度: beginner
🔧 相关技术: Py